In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 99 Utils — Console Logger
# MAGIC
# MAGIC **Notebook:** `utils_logger`
# MAGIC
# MAGIC Provides standardized console logging utilities for the Brazil Legislative Analytics Medallion pipeline.
# MAGIC
# MAGIC This notebook centralizes log formatting across Bronze, Silver, Gold, Marts, Quality and Jobs notebooks.
# MAGIC
# MAGIC ## Responsibilities
# MAGIC - Standardize operational logging across pipeline notebooks
# MAGIC - Support structured log levels such as INFO, WARNING, SUCCESS and ERROR
# MAGIC - Include Medallion layer context in log messages
# MAGIC - Improve execution traceability during Databricks notebook runs
# MAGIC - Support troubleshooting during API ingestion and transformation steps
# MAGIC
# MAGIC ## Technical Notes
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.
# MAGIC - This logger writes messages to the notebook execution output.
# MAGIC - Persistent logging is handled by `utils_table_logger`.

# COMMAND ----------

import logging
from typing import Optional

# COMMAND ----------

def get_logger(
    logger_name: str,
    layer_name: str = "pipeline",
) -> logging.Logger:
    """
    Creates or retrieves a standardized pipeline logger.
    """

    normalized_layer = layer_name.strip().lower()
    full_logger_name = f"{normalized_layer}.{logger_name}"

    pipeline_logger = logging.getLogger(full_logger_name)

    if pipeline_logger.handlers:
        return pipeline_logger

    pipeline_logger.setLevel(logging.INFO)
    pipeline_logger.propagate = False

    log_formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(layer)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(log_formatter)

    class LayerFilter(logging.Filter):
        """
        Adds the Medallion layer name to each log record.
        """

        def filter(self, log_record: logging.LogRecord) -> bool:
            log_record.layer = normalized_layer.upper()
            return True

    console_handler.addFilter(LayerFilter())
    pipeline_logger.addHandler(console_handler)

    return pipeline_logger

# COMMAND ----------

def log_info(
    pipeline_logger: logging.Logger,
    message: str,
) -> None:
    """
    Registers an informational console log message.
    """

    pipeline_logger.info(message)

# COMMAND ----------

def log_warning(
    pipeline_logger: logging.Logger,
    message: str,
) -> None:
    """
    Registers a warning console log message.
    """

    pipeline_logger.warning(message)

# COMMAND ----------

def log_error(
    pipeline_logger: logging.Logger,
    message: str,
    error: Optional[Exception] = None,
) -> None:
    """
    Registers an error console log message.
    """

    if error is not None:
        pipeline_logger.error(f"{message} | error_detail={str(error)}")
    else:
        pipeline_logger.error(message)

# COMMAND ----------

def log_success(
    pipeline_logger: logging.Logger,
    message: str,
) -> None:
    """
    Registers a success console log message.

    Notes
    -----
    Python logging does not provide a native SUCCESS level.
    For Databricks compatibility, SUCCESS is registered as INFO with a prefix.
    """

    pipeline_logger.info(f"[SUCCESS] {message}")

# COMMAND ----------

print("utils_logger loaded successfully.")

utils_logger loaded successfully.
